# MLP Knowledge Profiling

Profile what knowledge MLP neurons encode: vocabulary preferences,
selectivity, position specificity, cooperation, and cross-layer summary.

## Why This Matters

Knowledge mlp investigates where and how models store factual information. Locating knowledge in specific components (neurons, layers, directions) is essential for understanding hallucination, enabling fact editing, and assessing what a model truly knows.

**Key references:**
- [Geva et al. (2021) "Transformer Feed-Forward Layers Are Key-Value Memories"](https://arxiv.org/abs/2012.14913) — Shows MLPs function as key-value memory stores
- [Meng et al. (2022) "Locating and Editing Factual Associations"](https://arxiv.org/abs/2202.05262) — Causal tracing to locate factual knowledge

In [ ]:
import jax
import jax.numpy as jnp
from irtk import HookedTransformer, HookedTransformerConfig
from irtk.mlp_knowledge_profiling import (
    neuron_vocabulary_profile, neuron_selectivity_profile,
    neuron_position_specificity, neuron_cooperation_profile,
    layer_knowledge_summary,
)

cfg = HookedTransformerConfig(
    n_layers=4, d_model=32, n_ctx=64, d_head=8, n_heads=4, d_vocab=100,
)
model = HookedTransformer(cfg)
key = jax.random.PRNGKey(42)
leaves, treedef = jax.tree.flatten(model)
new_leaves = []
for leaf in leaves:
    if isinstance(leaf, jnp.ndarray) and leaf.dtype == jnp.float32:
        key, subkey = jax.random.split(key)
        new_leaves.append(jax.random.normal(subkey, leaf.shape) * 0.1)
    else:
        new_leaves.append(leaf)
model = jax.tree.unflatten(treedef, new_leaves)
tokens = jnp.array([1, 15, 30, 45, 60, 75, 90])
print('Model ready')

## Vocabulary Profile

Which tokens does each top neuron most promote or suppress?

In [ ]:
result = neuron_vocabulary_profile(model, layer=0, top_k=8)
for n in result['per_neuron']:
    print(f"  Neuron {n['neuron']:3d}: norm={n['output_norm']:.4f}, "
          f"promotes tok {n['top_promoted']:3d} ({n['top_promoted_logit']:+.4f}), "
          f"suppresses tok {n['top_suppressed']:3d} ({n['top_suppressed_logit']:+.4f}), "
          f"range={n['logit_range']:.4f}")

## Selectivity Profile

Which neurons activate on few tokens (selective) vs many (broad)?

In [ ]:
result = neuron_selectivity_profile(model, tokens, layer=0, top_k=8)
print(f"Ever active: {result['n_ever_active']}, Dead: {result['n_dead']}\n")
for n in result['per_neuron']:
    sel = 'SELECTIVE' if n['is_selective'] else 'broad'
    print(f"  Neuron {n['neuron']:3d}: rate={n['activation_rate']:.2f}, "
          f"mean_act={n['mean_activation']:.4f} [{sel}]")

## Position Specificity

Do MLP neurons activate differently at different positions?

In [ ]:
result = neuron_position_specificity(model, tokens, layer=0)
for p in result['per_position']:
    bar = '█' * int(p['sparsity'] * 20)
    print(f"  Pos {p['position']}: active={p['n_active']}, "
          f"sparsity={p['sparsity']:.2f}, max={p['max_activation']:.4f}, "
          f"top_neuron={p['top_neuron']} {bar}")

## Cooperation Profile

Do certain neurons consistently co-activate?

In [ ]:
result = neuron_cooperation_profile(model, tokens, layer=0, sample_size=30)
print(f"Cooperative pairs found: {result['n_cooperative_pairs']}\n")
for p in result['pairs'][:10]:
    print(f"  Neuron {p['neuron_a']:3d} & {p['neuron_b']:3d}: "
          f"coact={p['coactivation_rate']:.3f}, jaccard={p['jaccard']:.3f}")

## Layer Knowledge Summary

Cross-layer overview of MLP knowledge encoding activity.

In [ ]:
result = layer_knowledge_summary(model, tokens)
for p in result['per_layer']:
    bar = '█' * int(min(p['logit_impact'] * 50, 20))
    print(f"  Layer {p['layer']}: active={p['mean_active_neurons']}, "
          f"magnitude={p['mean_activation_magnitude']:.4f}, "
          f"logit_impact={p['logit_impact']:.4f} {bar}")